In [ ]:
import boto3
import awswrangler as wr
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configura a região padrão da AWS para o awswrangler
boto3.setup_default_session(region_name="us-east-1")

# Configuração do banco de dados
GLUE_DATABASE = "classicmodels_analytics_db"

# Configuração visual do Seaborn
sns.set_theme(style="whitegrid")

In [4]:
query_products = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

# Executa a query no Athena e retorna um DataFrame do Pandas diretamente
df_products_sample = wr.athena.read_sql_query(sql=query_products, database=GLUE_DATABASE)
df_products_sample.head()

c:\Users\gubia\OneDrive\Área de Trabalho\fgv-projetos-20261\venv\Lib\site-packages\awswrangler\athena\_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,product_id,product_name,product_line,product_vendor
0,S32_1374,1997 BMW F650 ST,Motorcycles,Exoto Designs
1,S24_2011,18th century schooner,Ships,Carousel DieCast Legends
2,S18_1889,1948 Porsche 356-A Roadster,Classic Cars,Gearbox Collectibles
3,S18_2432,1926 Ford Fire Engine,Trucks and Buses,Carousel DieCast Legends
4,S700_1938,The Mayflower,Ships,Studio M Art Models


In [6]:
query_sales_by_country = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_sales_country = wr.athena.read_sql_query(sql=query_sales_by_country, database=GLUE_DATABASE)
df_sales_country

c:\Users\gubia\OneDrive\Área de Trabalho\fgv-projetos-20261\venv\Lib\site-packages\awswrangler\athena\_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,country,total_sales
0,USA,3273280.05
1,Spain,1099389.09
2,France,1007374.02
3,Australia,562582.59
4,New Zealand,476847.01
5,UK,436947.44
6,Italy,360616.81
7,Finland,295149.35
8,Singapore,263997.78
9,Denmark,218994.92


In [7]:
query_cube = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

# Armazena a base analítica bruta no Pandas
df_analytics = wr.athena.read_sql_query(sql=query_cube, database=GLUE_DATABASE)

# Conversão explícita da coluna de data para o tipo datetime do Pandas
df_analytics['full_date'] = pd.to_datetime(df_analytics['full_date'])

print(f"Base analítica carregada com sucesso. Total de linhas: {len(df_analytics)}")

c:\Users\gubia\OneDrive\Área de Trabalho\fgv-projetos-20261\venv\Lib\site-packages\awswrangler\athena\_utils.py:839: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


Base analítica carregada com sucesso. Total de linhas: 2996


In [ ]:
# Determina os limites de data e as listas únicas para popular os componentes de filtro
min_date = df_analytics['full_date'].min().date()
max_date = df_analytics['full_date'].max().date()

paises_disponiveis = ['Todos'] + sorted(df_analytics['country'].unique().tolist())
linhas_disponiveis = ['Todos'] + sorted(df_analytics['product_line'].unique().tolist())

# Definição dos Widgets
date_picker_start = widgets.DatePicker(description='Data Início:', value=min_date)
date_picker_end = widgets.DatePicker(description='Data Fim:', value=max_date)
dropdown_country = widgets.Dropdown(options=paises_disponiveis, value='Todos', description='País:')
dropdown_line = widgets.Dropdown(options=linhas_disponiveis, value='Todos', description='Linha Prod:')
slider_top_n = widgets.IntSlider(value=5, min=1, max=10, step=1, description='Top N:')

# Container de saída para renderização do gráfico
output_plot = widgets.Output()

def atualizar_dashboard(change):
    with output_plot:
        clear_output(wait=True)
        
        # Filtro por intervalo de datas
        start_dt = pd.to_datetime(date_picker_start.value)
        end_dt = pd.to_datetime(date_picker_end.value)
        df_filtrado = df_analytics[(df_analytics['full_date'] >= start_dt) & (df_analytics['full_date'] <= end_dt)]
        
        # Filtro por país
        if dropdown_country.value != 'Todos':
            df_filtrado = df_filtrado[df_filtrado['country'] == dropdown_country.value]
            
        # Filtro por linha de produto
        if dropdown_line.value != 'Todos':
            df_filtrado = df_filtrado[df_filtrado['product_line'] == dropdown_line.value]
            
        # Ranqueamento
        df_agrupado = df_filtrado.groupby('product_name')['total_sales'].sum().reset_index()
        df_top_n = df_agrupado.sort_values(by='total_sales', ascending=False).head(slider_top_n.value)
        
        # Geração do gráfico
        if not df_top_n.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(
                x='total_sales', 
                y='product_name', 
                data=df_top_n, 
                ax=ax, 
                palette='viridis', 
                hue='product_name', 
                legend=False
            )
            ax.set_title(f"Top {slider_top_n.value} Produtos por Volume de Vendas", fontsize=14, pad=15)
            ax.set_xlabel("Vendas Totais ($)", fontsize=12)
            ax.set_ylabel("Nome do Produto", fontsize=12)
            plt.tight_layout()
            plt.show()
        else:
            print("Nenhum registro encontrado para os filtros selecionados.")

# Vincula o evento de alteração de valor dos componentes à função de atualização
date_picker_start.observe(atualizar_dashboard, names='value')
date_picker_end.observe(atualizar_dashboard, names='value')
dropdown_country.observe(atualizar_dashboard, names='value')
dropdown_line.observe(atualizar_dashboard, names='value')
slider_top_n.observe(atualizar_dashboard, names='value')

# Organização e exibição do layout em tela
controles_data = widgets.HBox([date_picker_start, date_picker_end])
controles_filtros = widgets.HBox([dropdown_country, dropdown_line, slider_top_n])
layout_final = widgets.VBox([controles_data, controles_filtros, output_plot])

display(layout_final)

# Dispara a renderização inicial do gráfico
atualizar_dashboard(None)